# Week 2a — Train ECAPA-TDNN From Scratch (100 speakers)

Trains only. Saves a checkpoint + model config that the separate evaluation notebook loads. This split lets you re-run evaluation (with different enrollment/test sweeps) without retraining.

**Scope:** ~100 speakers from VoxCeleb1 dev, random init, channels=512 (scaled-down ECAPA-TDNN), AAM-Softmax loss.

After this finishes: **Save Version** on this notebook (so its output is versioned), then attach it as an input dataset to the evaluation notebook.

In [ ]:
!pip install -q speechbrain


In [ ]:
import os, random, pickle, time, json
from pathlib import Path
import torch
import torch.nn as nn
import torchaudio
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
torch.manual_seed(42)
random.seed(42)


In [ ]:
path_split = Path("/kaggle/input/datasets/sabahesaraki/voxceleb-1-dataset")
path_data = Path("/kaggle/input/datasets/kryakrya")
path_data_train = path_data.joinpath("voxceleb1train/wav")
path_data_test  = path_data.joinpath("voxceleb1test/wav")
print("Paths OK.")


## 1. Select the 100-speaker training subset

In [ ]:
N_SPEAKERS = 100
N_VAL_PER_SPEAKER = 2

all_train_speakers = sorted([p.name for p in path_data_train.iterdir() if p.is_dir()])
print("Total available dev speakers:", len(all_train_speakers))

random.seed(42)
selected_speakers = sorted(random.sample(all_train_speakers, N_SPEAKERS))
speaker_to_label = {spk: i for i, spk in enumerate(selected_speakers)}
print(f"Selected {len(selected_speakers)} speakers for training.")

train_files, val_files = [], []
for spk in tqdm(selected_speakers, desc="Listing files"):
    spk_dir = path_data_train / spk
    wav_files = list(spk_dir.rglob("*.wav"))
    rel_paths = [str(w.relative_to(path_data_train)) for w in wav_files]
    random.shuffle(rel_paths)
    val_part = rel_paths[:N_VAL_PER_SPEAKER]
    train_part = rel_paths[N_VAL_PER_SPEAKER:]
    label = speaker_to_label[spk]
    train_files.extend([(p, label) for p in train_part])
    val_files.extend([(p, label) for p in val_part])

print("Train utterances:", len(train_files))
print("Val utterances:  ", len(val_files))


## 2. Dataset — random 3-second crops

In [ ]:
SAMPLE_RATE = 16000
CROP_SECONDS = 3.0
CROP_SAMPLES = int(SAMPLE_RATE * CROP_SECONDS)

class VoxCelebCropDataset(Dataset):
    def __init__(self, file_label_list, root):
        self.items = file_label_list
        self.root = root

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        rel_path, label = self.items[idx]
        full_path = self.root / rel_path
        signal, fs = torchaudio.load(str(full_path))
        if fs != SAMPLE_RATE:
            signal = torchaudio.functional.resample(signal, fs, SAMPLE_RATE)
        signal = signal.mean(dim=0)

        if signal.shape[0] >= CROP_SAMPLES:
            start = random.randint(0, signal.shape[0] - CROP_SAMPLES)
            signal = signal[start:start + CROP_SAMPLES]
        else:
            reps = CROP_SAMPLES // signal.shape[0] + 1
            signal = signal.repeat(reps)[:CROP_SAMPLES]

        return signal, label

train_ds = VoxCelebCropDataset(train_files, path_data_train)
val_ds = VoxCelebCropDataset(val_files, path_data_train)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

xb, yb = next(iter(train_loader))
print("Batch shapes:", xb.shape, yb.shape)


## 3. Model — small ECAPA-TDNN (random init) + AAM-Softmax head

In [ ]:
from speechbrain.lobes.models.ECAPA_TDNN import ECAPA_TDNN, Classifier
from speechbrain.lobes.features import Fbank
from speechbrain.processing.features import InputNormalization
from speechbrain.nnet.losses import LogSoftmaxWrapper, AdditiveAngularMargin

N_MELS = 80
EMB_DIM = 192
CHANNELS = [512, 512, 512, 512, 1536]
KERNEL_SIZES = [5, 3, 3, 3, 1]
DILATIONS = [1, 2, 3, 4, 1]
ATTENTION_CHANNELS = 128

compute_features = Fbank(n_mels=N_MELS).to(device)
mean_var_norm = InputNormalization(norm_type="sentence", std_norm=False).to(device)

embedding_model = ECAPA_TDNN(
    input_size=N_MELS,
    channels=CHANNELS,
    kernel_sizes=KERNEL_SIZES,
    dilations=DILATIONS,
    groups=[1, 1, 1, 1, 1],
    attention_channels=ATTENTION_CHANNELS,
    lin_neurons=EMB_DIM,
).to(device)

classifier = Classifier(
    input_size=EMB_DIM,
    out_neurons=N_SPEAKERS,
).to(device)

n_params = sum(p.numel() for p in embedding_model.parameters())
print(f"Embedding model params: {n_params/1e6:.2f}M")

loss_fn = LogSoftmaxWrapper(loss_fn=AdditiveAngularMargin(margin=0.2, scale=30))

def extract_features(signal_batch):
    feats = compute_features(signal_batch)
    lens = torch.ones(feats.shape[0], device=device)
    feats = mean_var_norm(feats, lens)
    return feats


## 4. Training loop (per-epoch checkpointing)

In [ ]:
N_EPOCHS = 30
LR = 0.001

params = list(embedding_model.parameters()) + list(classifier.parameters())
optimizer = torch.optim.Adam(params, lr=LR)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, total_steps=N_EPOCHS * len(train_loader)
)

CKPT_DIR = Path("/kaggle/working/checkpoints")
CKPT_DIR.mkdir(exist_ok=True)

history = {"train_loss": [], "train_acc": [], "val_acc": []}

def run_epoch(loader, train=True):
    embedding_model.train(train)
    classifier.train(train)
    total_loss, total_correct, total_n = 0.0, 0, 0
    for signal, label in loader:
        signal, label = signal.to(device), label.to(device)
        with torch.set_grad_enabled(train):
            feats = extract_features(signal)
            emb = embedding_model(feats)
            logits = classifier(emb)
            loss = loss_fn(logits, label.unsqueeze(1))
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                scheduler.step()

        preds = logits.squeeze(1).argmax(dim=1)
        total_correct += (preds == label).sum().item()
        total_n += label.size(0)
        total_loss += loss.item() * label.size(0)

    return total_loss / total_n, total_correct / total_n


In [ ]:
for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    dt = time.time() - t0

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:02d}/{N_EPOCHS}  train_loss={train_loss:.4f}  "
          f"train_acc={train_acc*100:.1f}%  val_acc={val_acc*100:.1f}%  ({dt:.1f}s)")

    torch.save({
        'embedding_model': embedding_model.state_dict(),
        'classifier': classifier.state_dict(),
        'epoch': epoch,
        'history': history,
    }, CKPT_DIR / f"ckpt_epoch{epoch}.pt")
    prev = CKPT_DIR / f"ckpt_epoch{epoch-1}.pt"
    if prev.exists():
        prev.unlink()

# Save a stable-named final checkpoint (easier for the eval notebook to find regardless of epoch count)
last_ckpt = CKPT_DIR / f"ckpt_epoch{N_EPOCHS}.pt"
final_path = CKPT_DIR / "final_checkpoint.pt"
torch.save(torch.load(last_ckpt), final_path)
print("Training complete. Final checkpoint saved to:", final_path)


In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history["train_loss"])
plt.title("Train Loss"); plt.xlabel("Epoch"); plt.grid(alpha=0.3)

plt.subplot(1,2,2)
plt.plot(history["train_acc"], label="train")
plt.plot(history["val_acc"], label="val")
plt.title("Speaker Classification Accuracy"); plt.xlabel("Epoch"); plt.legend(); plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/scratch_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Save model config (needed by the evaluation notebook to rebuild the exact architecture)

In [ ]:
model_config = {
    "n_speakers": N_SPEAKERS,
    "n_mels": N_MELS,
    "emb_dim": EMB_DIM,
    "channels": CHANNELS,
    "kernel_sizes": KERNEL_SIZES,
    "dilations": DILATIONS,
    "attention_channels": ATTENTION_CHANNELS,
    "sample_rate": SAMPLE_RATE,
    "n_epochs_trained": N_EPOCHS,
    "selected_speakers": selected_speakers,
}
with open("/kaggle/working/checkpoints/model_config.json", "w") as f:
    json.dump(model_config, f, indent=2)

print("Saved model_config.json")
print("\n=== NEXT STEP ===")
print("1. Click 'Save Version' on this notebook (Save & Run All, or Quick Save).")
print("2. Once versioned, its output (checkpoints/final_checkpoint.pt + model_config.json)")
print("   becomes available as a Kaggle Dataset you can attach to the evaluation notebook.")
